# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import certifi
import duckdb
from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()


get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

In [2]:
import re

_con = None

def get_duckdb_connection():
    global _con
    if _con is not None:
        return _con

    _con = duckdb.connect()
    _con.execute("INSTALL delta;")
    _con.execute("LOAD delta;")

    if str(DATA_DIR).startswith('s3://'):
        _con.execute("INSTALL httpfs;")
        _con.execute("LOAD httpfs;")


        _con.execute(
            "CREATE OR REPLACE SECRET emaps_s3 (TYPE S3, KEY_ID ?, SECRET ?, REGION ?)",
            [
                settings.aws_access_key_id,
                settings.aws_secret_access_key,
                settings.aws_region or "ap-south-1",
            ],
        )

    return _con

def print_sql_df(sql_query):
    con = get_duckdb_connection()
    
    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)
        
        if schema == "bronze":
            return f"{keyword} read_parquet('{DATA_DIR}/{schema}/{table}/**/*.parquet')"
        else:
            return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)', 
        replace_table, 
        sql_query
    )
    df = con.execute(parsed_query).df()
    display(df)

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows limit 10')

Bronze Daily Flows Summary:


,raw_json,record_count,day,month,year
0,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix limit 10')

Bronze Daily Mix Summary:


,raw_json,record_count,day,month,year
0,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [5]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports limit 10')

Gold Daily Exports:


,zone,zone_metadata,destination_zone,reference_timestamp,net_mwh,hours_covered,year,month
0,FR,France,GB,2026-04-25,31686.0,11,2026,4
1,FR,France,LU,2026-04-24,2399.0,13,2026,4
2,FR,France,GB,2026-04-24,39924.0,13,2026,4
3,FR,France,IT-NO,2026-04-25,13621.0,12,2026,4
4,FR,France,IT-NO,2026-04-24,45087.0,13,2026,4
5,FR,France,LU,2026-04-25,2147.0,11,2026,4


In [6]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports LIMIT 10")


Gold Daily Imports:


,zone,zone_metadata,source_zone,reference_timestamp,net_mwh,hours_covered,year,month
0,FR,France,BE,2026-04-25,36944.0,11,2026,4
1,FR,France,DE,2026-04-25,14236.0,11,2026,4
2,FR,France,CH,2026-04-24,16526.0,13,2026,4
3,FR,France,CH,2026-04-25,7386.0,11,2026,4
4,FR,France,ES,2026-04-24,30674.0,13,2026,4
5,FR,France,BE,2026-04-24,38216.0,13,2026,4
6,FR,France,ES,2026-04-25,30819.0,11,2026,4
7,FR,France,DE,2026-04-24,23532.0,13,2026,4


In [7]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix LIMIT 10")


Gold Daily Mix:


,zone,zone_metadata,reference_timestamp,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,oil_pct,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month
0,FR,France,2026-04-25,76.988908,2.198201,4.123178,6.961056,8.990026,0.668689,0.069942,0.0,0.0,0.0,585938.196,99.261370,22.272462,11,2026,4
1,FR,France,2026-04-24,71.596802,1.985558,6.479500,10.000255,9.249980,0.626422,0.061482,0.0,0.0,0.0,783242.301,99.312096,27.715293,13,2026,4


In [8]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777118445020,2026-04-25 17:30:46.599915+05:30,2026-04-25 17:30:51.172857+05:30,R,16,NaN
1,gold,1777118374506,2026-04-25 17:29:36.090072+05:30,NaT,I,<NA>,Field zone_metadata not found in schema
2,gold,1777118330305,2026-04-25 17:28:51.812250+05:30,NaT,I,<NA>,"unable to find column ""zone_name""; valid colum..."
3,gold,1777117842838,2026-04-25 17:20:44.642769+05:30,2026-04-25 17:20:49.367946+05:30,R,19,NaN
4,silver,1777117722575,2026-04-25 17:18:44.292626+05:30,2026-04-25 17:18:47.632944+05:30,C,193,NaN
5,bronze,1777117713554,2026-04-25 17:18:33.554399+05:30,2026-04-25 17:18:35.862466+05:30,C,48,NaN
